### Lặp Gauss-Seidel

### Yêu cầu trình bày (Ghi vào bài thi)
**1. Điều kiện hội tụ:**
Ma trận $A$ chéo trội hàng:
$$|A_{ii}| > \sum_{j \neq i} |A_{ij}| \quad \forall i$$

**2. Công thức lặp Gauss-Seidel:**
$$x_i^{(k+1)} = \frac{1}{A_{ii}} \left( b_i - \sum_{j=1}^{i-1} A_{ij} x_j^{(k+1)} - \sum_{j=i+1}^n A_{ij} x_j^{(k)} \right)$$
*(Nghĩa là: nghiệm $x_i$ ở bước mới nhất được tính bằng cách dùng ngay các $x_j$ mới nhất vừa tính được ở cùng bước)*


In [3]:
import numpy as np
from IPython.display import display, Math, Markdown

def _mat(M, p=5):
    """Chuyển đổi ma trận hoặc mảng thành chuỗi LaTeX bmatrix."""
    if hasattr(M[0], "__len__"):
        rows = " \\\\ ".join([" & ".join([f"{x:.{p}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"
    inner = " \\\\ ".join([f"{x:.{p}f}" for x in M])
    return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}"

def Gauss_Seidel_Ax_B(A_input, b_input, max_iter=200, epsilon=1e-5, x0=None):
    A = np.array(A_input, dtype=float)
    b = np.array(b_input, dtype=float).flatten()
    n = len(b)
    
    if x0 is None:
        X_k = np.zeros(n)
    else:
        X_k = np.array(x0, dtype=float).flatten()
        
    display(Markdown("## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (HỆ $Ax = b$)"))
    
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)
    
    D_inv = np.linalg.inv(D)
    B = -D_inv @ (L + U)
    d = D_inv @ b
    
    # -----------------------------------------------------------------
    # 1. KIỂM TRA ĐIỀU KIỆN HỘI TỰ VÀ ĐÁNH GIÁ SỐ DƯ
    # -----------------------------------------------------------------
    display(Markdown("### 1. Kiểm tra điều kiện hội tụ"))
    
    is_row_dom = True
    for i in range(n):
        if sum(abs(A[i, j]) for j in range(n) if j != i) >= abs(A[i,i]):
            is_row_dom = False
            break
            
    is_col_dom = True
    for j in range(n):
        if sum(abs(A[i, j]) for i in range(n) if i != j) >= abs(A[j,j]):
            is_col_dom = False
            break
            
    if is_row_dom:
        display(Markdown("Ma trận A **chéo trội hàng**, phương pháp lặp chắc chắn hội tụ."))
        s = 0.0
        q = 0.0
        for i in range(n):
            sum_l = sum(abs(A[i, j]) for j in range(i))
            sum_u = sum(abs(A[i, j]) for j in range(i+1, n))
            q_i = sum_l / (abs(A[i,i]) - sum_u) if (abs(A[i,i]) - sum_u) > 0 else float('inf')
            q = max(q, q_i)
        display(Math(f"s = {s:.5f}, \\quad q = \\max_i \\frac{{\\sum_{{j<i}} |a_{{ij}}|}}{{|a_{{ii}}| - \\sum_{{j>i}} |a_{{ij}}|}} \\approx {q:.5f}"))
        
    elif is_col_dom:
        display(Markdown("Ma trận A **chéo trội cột**, phương pháp lặp chắc chắn hội tụ."))
        s = 0.0
        q = 0.0
        for j in range(n):
            sum_u = sum(abs(A[i, j]) for i in range(j+1, n))
            s = max(s, sum_u / abs(j,j) if abs(A[j,j]) != 0 else float('inf'))
            sum_l = sum(abs(A[i, j]) for i in range(j))
            q_j = sum_l / (abs(A[j,j]) - sum_u) if (abs(A[j,j]) - sum_u) > 0 else float('inf')
            q = max(q, q_j)
        display(Math(f"s = \\max_j \\frac{{1}}{{|a_{{jj}}|}} \\sum_{{i>j}} |a_{{ij}}| \\approx {s:.5f}, \\quad q = \\max_j \\frac{{\\sum_{{i<j}} |a_{{ij}}|}}{{|a_{{jj}}| - \\sum_{{i>j}} |a_{{ij}}|}} \\approx {q:.5f}"))
        
    else:
        display(Markdown("**Cảnh báo:** Ma trận A không chéo trội ngặt hàng/cột. Phương pháp lặp có thể không hội tụ."))
        s = 0.0
        q = 0.0
        for i in range(n):
            sum_l = sum(abs(A[i, j]) for j in range(i))
            sum_u = sum(abs(A[i, j]) for j in range(i+1, n))
            q_i = sum_l / (abs(A[i,i]) - sum_u) if (abs(A[i,i]) - sum_u) > 0 else float('inf')
            if q_i != float('inf'): q = max(q, q_i)
        if q >= 1: q = 0.99
        display(Math(f"s = {s:.5f}, \\quad q \\approx {q:.5f}"))
        
    # Xử lý an toàn khi epsilon nhận giá trị None
    if epsilon is not None:
        eps0 = epsilon * (1 - s) * (1 - q) / q if q < 1 and q > 0 else epsilon
        display(Math(f"\\varepsilon_0 = \\frac{{\\varepsilon(1-s)(1-q)}}{{q}} = {eps0:.5e}"))
    else:
        eps0 = None
        display(Markdown("Sai số $\\varepsilon = \\text{None}$: Thuật toán chỉ dừng theo số lần lặp tối đa (`max_iter`)."))
    
    # -----------------------------------------------------------------
    # 2. QUÁ TRÌNH TÍNH TOÁN SAI SỐ QUA TỪNG VÒNG LẶP
    # -----------------------------------------------------------------
    display(Markdown("### 2. Quá trình lặp"))
    
    history = []
    diffs = []
    k = 1
    
    while True:
        X_new = np.copy(X_k)
        for i in range(n):
            s1 = sum(B[i, j] * X_new[j] for j in range(i))
            s2 = sum(B[i, j] * X_k[j] for j in range(i, n))
            X_new[i] = s1 + s2 + d[i]
            
        diff = np.linalg.norm(X_new - X_k, np.inf)
        history.append(X_new.copy())
        diffs.append(diff)
        
        # Tạo cờ logic kiểm soát điều kiện dừng
        stop_by_eps = (eps0 is not None and diff < eps0)
        stop_by_iter = (max_iter is not None and k >= max_iter)
        
        if stop_by_eps or stop_by_iter or k > 200:
            break
            
        X_k = np.copy(X_new)
        k += 1
        
    # -----------------------------------------------------------------
    # 3. THIẾT KẾ BẢNG HIỂN THỊ HÀNG NGANG KHÔNG LỖI ĐỊNH DẠNG
    # -----------------------------------------------------------------
    N = len(history)
    if N <= 5:
        cols = list(range(1, N+1))
    else:
        cols = [1, 2, -1, N-1, N]
        
    # Tạo tiêu đề cột động dựa trên số lượng biến hệ phương trình n
    header = "| Lần lặp | " + " | ".join([f"$x_{{{i+1}}}$" for i in range(n)]) + " | $|| x_k - x_{k-1} ||_\\infty$ |"
    sep = "|---|:" + ":|:".join(["---" for _ in range(n)]) + ":|:---:|"
    lines = [header, sep]
    
    for c in cols:
        if c == -1:
            row_dots = ["..."] + ["..."] * n + ["..."]
            lines.append("| " + " | ".join(row_dots) + " |")
        else:
            row = [f"Lần {c}"]
            for i in range(n):
                row.append(f"{history[c-1][i]:.5f}")
            
            val_str = f"{diffs[c-1]:.5f}"
            if eps0 is not None:
                if c == N:
                    val_str += f" < \\varepsilon_0"
                elif c == N-1:
                    val_str += f" > \\varepsilon_0"
            row.append(val_str)
            lines.append("| " + " | ".join(row) + " |")
            
    display(Markdown("\n".join(lines)))
    
    # -----------------------------------------------------------------
    # 4. KẾT LUẬN NGHIỆM CUỐI CÙNG
    # -----------------------------------------------------------------
    display(Markdown("### 3. Kết luận"))
    display(Markdown(f"Nghiệm xấp xỉ hệ phương trình sau {N} lần lặp:"))
    display(Math(f"X \\approx {_mat(history[-1], 5)}"))

# ═══════════════════════════════════════════════════════════════════
# DỮ LIỆU ĐỀ BÀI KHỞI TẠO ĐẦU VÀO
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [10.0, 1.0,  2.0],
    [ 1.0, 10.0, 3.0],
    [ 2.0,  3.0, 10.0]
], dtype=float)

b = np.array([7.0, 8.0, 9.0], dtype=float)
# ═══════════════════════════════════════════════════════════════════

# Kiểm thử thuật toán lặp dừng theo chu kỳ cố định
Gauss_Seidel_Ax_B(A, b, max_iter=5, epsilon=None)

## ❖ PHƯƠNG PHÁP LẶP GAUSS-SEIDEL (HỆ $Ax = b$)

### 1. Kiểm tra điều kiện hội tụ

Ma trận A **chéo trội hàng**, phương pháp lặp chắc chắn hội tụ.

<IPython.core.display.Math object>

Sai số $\varepsilon = \text{None}$: Thuật toán chỉ dừng theo số lần lặp tối đa (`max_iter`).

### 2. Quá trình lặp

| Lần lặp | $x_{1}$ | $x_{2}$ | $x_{3}$ | $|| x_k - x_{k-1} ||_\infty$ |
|---|:---:|:---:|:---:|:---:|
| Lần 1 | 0.70000 | 0.73000 | 0.54100 | 0.73000 |
| Lần 2 | 0.51880 | 0.58582 | 0.62049 | 0.18120 |
| Lần 3 | 0.51732 | 0.56212 | 0.62790 | 0.02370 |
| Lần 4 | 0.51821 | 0.55981 | 0.62842 | 0.00231 |
| Lần 5 | 0.51834 | 0.55964 | 0.62844 | 0.00017 |

### 3. Kết luận

Nghiệm xấp xỉ hệ phương trình sau 5 lần lặp:

<IPython.core.display.Math object>